# WHW04 starter notebook: solving an ODE symbolically and numerically

We're going to solve 

$$\frac{dy}{dt} = -3y, \qquad y(0) = 1$$

with SymPy, and then we're going to solve by making numerical approximations. The numerical method we'll use is called **forward Euler**.

First, let's use SymPy, like we did in the earlier notebooks.

## 0. Setup

Here's where we do all of our python setup. We'll grab matplotlib for plotting, sympy for symbolic manipulation, and numpy for numerical python calculations

In [ ]:
%matplotlib inline
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from sympy import *
init_printing()

## 1. The exact solution

Use SymPy's `dsolve` to solve $dy/dt = -3y$ with $y(0) = 1$, and confirm the
answer with `checkodesol`. 

Remember, we've done this in (`S26_Sympy_ODE_basics.ipynb`).

Work in the empty cells below. Keep the symbol for time named `t`, as we'll use it later on.

Call your solution `solution`, as we'll use that when we plot things later on.

In [ ]:
t = sp.symbols('t')

# your dsolve and checkodesol work here

## 2. Plotting

Let's plot our solution using matplotlib.

Because I know what the answer should look like, I'll tell you that we'll get good results plotting things from t = 0 to 6. In order to do that, we tell `numpy` to make an array of linearly spaced points between 0 and 6. Let's make 400 points in that range. That'll give us a nice smooth plot. We do that with

```python
t_smooth = np.linspace(0, 6, 400)
```

We then use matplotlib to plot things. That means we have to tell our SymPy solution to interact with actual numbers. SymPy comes with helper functions to do that. We'll use the one called `lambdify`. That's the part that says

```python
solution_numpy = sp.lambdify(t, solution.rhs, 'numpy')
```

Note that the solution has a left hand side and a right hand side. The right hand side (you printed the solution out, right?) is the actual equation we want, so we call `solution.rhs` above.

After that, we tell matplotlib to plot things. We'll be content with a fairly simple plot for now.

In [ ]:
t_smooth = np.linspace(0, 6, 400)
solution_numpy = sp.lambdify(t, solution.rhs, 'numpy')

# "label" tells it how to label the the line when
# we call legend later on.
plt.plot(t_smooth, solution_numpy(t_smooth),
         label=fr'${latex(solution)}$')
plt.xlabel('t')
plt.legend();

## 3. Numerical approximation (setup)

The numerical method we're using is called **forward Euler**:

You start at $(t_n, y_n)$. You want to figure out where a nearby point
to the right, $(t_{n+1}, y_{n+1})$ is. One way is to compute the
slope at your point ($\frac{dy}{dt}(t_n, y_n)$) and walk along that slope for a time step $h$:

$$y_{n+1} = y_n + h\, \frac{dy}{dt}(t_n, y_n), \qquad t_{n+1} = t_n + h .$$

Repeat, and you have the solution as a list of numbers, one per step. For our
equation, $\frac{dy}{dt} = -3y$ by definition.

This notebook shows the NumPy you need on a *different* equation, then leaves
the homework equation to you.

## 4. Predict first (before you run anything)

1. As the step size $h$ grows, we expect things to get worse. Does the Euler solution just get *less accurate*,
   or can it go qualitatively wrong? If wrong, in what way?
2. The exact solution decays smoothly to zero. Sketch what you expect an Euler
   solution with a large step to look like.

Double-click the cell below, type your prediction, and press Shift-Enter.

*(your prediction here)*

## 5. Numerical approximation (let's do it!)

Here is forward Euler, start to finish, for an equation whose answer we already
know:

$$\frac{dy}{dt} = \cos t, \qquad y(0) = 0, \qquad \text{exact: } y = \sin t .$$

For convenience, we'll call $\frac{dy}{dt}$ by a simpler-to-type name $f(t, y)$. We'll refer to it as our slope function. For this equation the slope depends only on $t$.

In [ ]:
def slope_demo(t_now, y_now):
    return np.cos(t_now)

Now the loop. Here's a line-by-line explanation. Again, you should read through this and understand it ... and then feel free to copy and modify it for your own purposes.

- `n_steps` is how many steps of size `h` it takes to reach `t_end`.
- `np.zeros(n_steps + 1)` makes an array of that many zeros, one slot per time,
  including the starting time. We'll fill the slots in as we go.
- `t_vals[0]` and `y_vals[0]` are the starting point; Python counts from 0.
- `for n in range(n_steps):` runs the indented lines with `n = 0, 1, 2, ...`,
  stopping one short of `n_steps`. That works for us: step `n` fills slot `n + 1`.

In [ ]:
def euler_demo(h, t_end):
    n_steps = int(round(t_end / h))
    t_vals = np.zeros(n_steps + 1)
    y_vals = np.zeros(n_steps + 1)
    t_vals[0] = 0.0
    y_vals[0] = 0.0
    for n in range(n_steps):
        y_vals[n + 1] = y_vals[n] + h * slope_demo(t_vals[n], y_vals[n])
        t_vals[n + 1] = t_vals[n] + h
    return t_vals, y_vals

Now, run it with two step sizes and look at the numbers:

In [ ]:
t_coarse, y_coarse = euler_demo(0.5, 6.0)
t_fine, y_fine = euler_demo(0.1, 6.0)
print(t_coarse)
print(y_coarse)

## 6. Comparing with an exact answer

We learned a bit about how to plot things earlier. Let's do the same thing here.

First, we use `sp.lambdify` to turn an exact equation into something that can talk to numpy.

Then we make our list of times, at which we'll evaluate things.

Then we plot. The plot is a bit fancier now, as we tell it the line style for each of the different lines.

In [ ]:
exact_demo = sp.sin(t)
exact_demo_numpy = sp.lambdify(t, exact_demo, 'numpy')

t_smooth = np.linspace(0, 6, 400)
plt.plot(t_smooth, exact_demo_numpy(t_smooth), 'k', label='exact: sin t')
plt.plot(t_coarse, y_coarse, 'o-', label='Euler, h = 0.5')
plt.plot(t_fine, y_fine, '.-', label='Euler, h = 0.1')
plt.xlabel('t')
plt.ylabel('y')
plt.legend()
plt.show()

## 7. Your turn: do it for our main ODE

You want to plot things for 

$dy/dt = -3y$, $y(0) = 1$, on $0 \le t \le 2$

Your slope function is the **right-hand side of the ODE**, $f(t, y)$ in $dy/dt = f(t, y)$.
It is *not* the exact solution: the whole point of Euler is that you don't know the solution,
only how steep it is at each point. (In the demo, $f = \cos t$ depended on $t$ alone; ours depends on $y$.)

Pretend your ODE were $dy/dt = 5ty$ (**NOTE** it most certainly is NOT that!). The numpy version of that
right-hand side is `5 * t_now * y_now`. So, if that had been your ODE, you'd write

```python
def my_slope_function(t_now, y_now):
    return 5 * t_now * y_now
```

For the Euler function, we can copy and paste almost everything from above. Two lines change: the
starting value `y_vals[0]` (our ODE starts at $y(0) = 1$), and the line that calls the slope function.

So, go for it in the cell below:

1. Modify `my_slope_function` to return the right-hand side of our ODE.
2. Modify `my_euler` to start from $y(0) = 1$ and to call `my_slope_function` instead of `slope_demo`.


3. Run it with $h = 0.1$, $h = 0.5$, and $h = 0.8$. Careful: $2/0.8$ is not a whole
   number of steps. Look at where your $h = 0.8$ run actually stops, and decide what
   you want it to do.
4. Plot all three together with your exact solution from section 1 (`solution_numpy`), on one
   set of axes: section 8 does this for you.

In [ ]:
# your slope function and Euler function here

def my_slope_function(t_now, y_now):
    # Change the line below!
    return np.cos(t_now)

def my_euler(h, t_end):
    n_steps = int(round(t_end / h))
    t_vals = np.zeros(n_steps + 1)
    y_vals = np.zeros(n_steps + 1)
    t_vals[0] = 0.0
    # Change the line below.
    y_vals[0] = 0.0
    for n in range(n_steps):
        # Change the line below.
        y_vals[n + 1] = y_vals[n] + h * slope_demo(t_vals[n], y_vals[n])
        t_vals[n + 1] = t_vals[n] + h
    return t_vals, y_vals

## 8. Now let's plot it and see if we were right.

First, we'll run the new functions with different values of h, 0.1, 0.5 and 0.8.

Then we'll plot things!


In [ ]:
t_smooth = np.linspace(0, 2, 400)
t_super_coarse, y_super_coarse = my_euler(0.8, 2.0)
t_coarse, y_coarse = my_euler(0.5, 2.0)
t_fine, y_fine = my_euler(0.1, 2.0)


plt.plot(t_smooth, solution_numpy(t_smooth), 'k', label='exact')
plt.plot(t_super_coarse, y_super_coarse, 's-', label='Euler, h = 0.8')
plt.plot(t_coarse, y_coarse, 'o-', label='Euler, h = 0.5')
plt.plot(t_fine, y_fine, '.-', label='Euler, h = 0.1')
plt.xlabel('t')
plt.ylabel('y')
plt.legend()
plt.show()

## 9. What you found

**(a)** Which of your predictions does the picture support? Two or three sentences.

*(your answer here)*

**(b)** Work out algebraically the largest $h$ for which your Euler solution still
decays toward zero, and check that number against your plot.

*(your answer here)*

**(c)** One thing that did not work along the way (an off-by-one in the loop, an
empty plot, a shape mismatch, ...).

*(your answer here)*